# Feature engineering: a hands-on implementation

**Track:** Machine learning · **Guide:** [`015-feature-engineering.md`](../guides/concepts/015-feature-engineering.md)

Feature engineering transforms raw fields into useful model inputs while preserving meaning and preventing leakage.

> **Intuition:** It turns raw ingredients into a form a particular recipe can use without accidentally adding the answer.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Feature engineering]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Build time-safe features for failure prediction and review them for target leakage and unstable definitions.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Identify numeric and categorical fields

Transformations should be defined from the task and available at prediction time, not from future outcomes.


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = Pipeline([("impute", SimpleImputer()), ("scale", StandardScaler())])
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

## 2. Fit transformations within the model pipeline

The pipeline ensures imputers, scalers, and category vocabularies are fitted on training data only.


In [ ]:
from sklearn.linear_model import LogisticRegression

preprocess = ColumnTransformer([
    ("number", numeric, [0, 1]),
    ("category", categorical, [2]),
])
pipeline = Pipeline([("features", preprocess), ("model", LogisticRegression())])
X = np.array([[20.0, 0.1, "north"], [25.0, 0.3, "south"], [None, 0.2, "north"], [24.0, 0.4, "south"]], dtype=object)
y = [0, 1, 0, 1]
pipeline.fit(X, y)

## 3. Handle unseen categories safely

A production pipeline must accept expected missing values and unseen categories while preserving a stable feature contract.


In [ ]:
prediction = pipeline.predict(np.array([[22.0, 0.2, "west"]], dtype=object))
print("prediction for unseen category:", prediction.tolist())

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/015-feature-engineering.md) and the
[learning roadmap](../guides/ROADMAP.md).
